# Text Vectorization Practice

**목표**
1. `ratings.txt` 파일을 불러와 텍스트 데이터를 준비할 수 있다.
2. `Okt`를 사용해 한국어 형태소 기반 토큰 전처리를 수행할 수 있다.
3. `CountVectorizer`와 `TfidfVectorizer`로 문서 벡터를 생성할 수 있다.
4. `Word2Vec`과 `FastText`로 단어 임베딩을 학습하고 결과를 비교할 수 있다.

## 0. 실습 환경 준비

In [ ]:
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec, FastText

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_colwidth', 120)

okt = Okt()

## 1. 실제 리뷰 데이터 불러오기

1. `ratings.txt` 파일을 읽어 `df_raw` 데이터프레임을 생성하기
2. 구분자는 `	` 로 설정하기
3. 결측치를 제거하기
4. `document` 컬럼만 사용하여 `df` 데이터프레임을 만들기
5. 너무 오래 걸리지 않도록 무작위로 3000개 샘플을 추출하기
6. 인덱스를 재설정하기
7. 앞부분 5개 행을 확인하기

In [ ]:
data_path = Path('ratings.txt')
df_raw = pd.read_csv(data_path, sep='	')
df_raw = df_raw.dropna(subset=['document'])
df = df_raw[['document']].sample(n=3000, random_state=42).reset_index(drop=True)

df.head()

## 2. 리뷰 길이와 기본 분포 확인하기

1. 문자 수를 저장하는 `char_len` 컬럼 만들기
2. 공백 기준 단어 수를 저장하는 `word_space_len` 컬럼 만들기
3. 두 컬럼의 기술통계량 확인하기
4. `char_len` 분포를 히스토그램으로 시각화하기

In [ ]:
df['char_len'] = df['document'].str.len()
df['word_space_len'] = df['document'].str.split().str.len()

print(df['char_len'].describe())
print(df['word_space_len'].describe())

plt.figure(figsize=(8, 4))
plt.hist(df['char_len'], bins=30)
plt.title('리뷰 문자 수 분포')
plt.xlabel('문자 수')
plt.ylabel('리뷰 수')
plt.show()

## 3. 정규표현식으로 리뷰 텍스트 정제하기

1. 한글과 공백만 남기고 나머지는 제거하는 `clean_text_fn()` 함수 만들기
2. 연속된 공백을 하나의 공백으로 정리하기
3. 양쪽 공백을 제거하기
4. 정제 결과를 저장하는 `clean_document` 컬럼 만들기
5. 원문과 정제문을 앞부분 5개 행으로 확인하기

In [ ]:
def clean_text_fn(text):
    text = str(text)
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_document'] = df['document'].apply(clean_text_fn)

df[['document', 'clean_document']].head()

## 4. Okt 기반 토큰 전처리 함수 만들기

1. 불용어 리스트 `ko_stopwords` 만들기
2. `Okt()` 객체를 사용해 형태소를 추출하기
3. `morphs(..., stem=True)`를 사용하기
4. 길이가 2 이상인 토큰만 남기기
5. 불용어를 제거하기
6. 토큰 리스트를 반환하는 `tokenize_ko_text()` 함수 만들기
7. 결과를 `tokens` 컬럼에 저장하기

In [ ]:
ko_stopwords = {
    '영화', '정말', '진짜', '너무', '아주', '그냥', '보고', '보다', '봤다', '봤는데',
    '이거', '이건', '이런', '저런', '있는', '없는', '하다', '하고', '에서', '이다',
    '같다', '수', '것', '더', '좀', '잘', '왜', '또', '내', '나', '만', '듯', '때',
    '데', '듯하다', '이다', '되다', '있다', '없다', '하다', '이다', '그리고', '하지만'
}

def tokenize_ko_text(text):
    morphs = okt.morphs(text, stem=True)
    tokens = [token for token in morphs if len(token) >= 2 and token not in ko_stopwords]
    return tokens

df['tokens'] = df['clean_document'].apply(tokenize_ko_text)

df[['clean_document', 'tokens']].head()

## 5. 전처리 결과 점검하기

1. `token_len` 컬럼을 만들어 토큰 개수를 저장하기
2. 토큰이 3개 이상인 리뷰만 남긴 `df_tokens` 데이터프레임 만들기
3. `df_tokens`의 크기 확인하기
4. 원문, 정제문, 토큰 결과를 5개 행 확인하기
5. 가장 자주 등장한 토큰 20개를 구하기

In [ ]:
df['token_len'] = df['tokens'].apply(len)
df_tokens = df[df['token_len'] >= 3].reset_index(drop=True)

print(df_tokens.shape)
print(df_tokens[['document', 'clean_document', 'tokens']].head())

all_tokens = [token for tokens in df_tokens['tokens'] for token in tokens]
top20_tokens = Counter(all_tokens).most_common(20)
print(top20_tokens)

## 6. CountVectorizer로 BoW 문서-단어 행렬 만들기

1. 토큰 리스트를 공백으로 연결한 `token_text` 컬럼 만들기
2. `CountVectorizer(max_features=1000)` 객체 생성하기
3. `bow_matrix` 생성하기
4. vocabulary 크기 확인하기
5. 상위 15개 단어의 전체 출현 빈도를 구하기

In [ ]:
df_tokens['token_text'] = df_tokens['tokens'].apply(lambda x: ' '.join(x))

count_vectorizer = CountVectorizer(max_features=1000)
bow_matrix = count_vectorizer.fit_transform(df_tokens['token_text'])

print('vocabulary size:', len(count_vectorizer.vocabulary_))

bow_word_sums = np.asarray(bow_matrix.sum(axis=0)).flatten()
bow_vocab = np.array(count_vectorizer.get_feature_names_out())
bow_top_idx = bow_word_sums.argsort()[::-1][:15]
top15_bow_words = list(zip(bow_vocab[bow_top_idx], bow_word_sums[bow_top_idx]))
print(top15_bow_words)

## 7. TfidfVectorizer로 TF-IDF 문서 벡터 만들기

1. `TfidfVectorizer(max_features=1000)` 객체 생성하기
2. `tfidf_matrix` 생성하기
3. vocabulary 크기 확인하기
4. TF-IDF 합계 기준 상위 15개 단어를 구하기
5. BoW 결과와 어떤 차이가 있는지 확인하기

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_tokens['token_text'])

print('vocabulary size:', len(tfidf_vectorizer.vocabulary_))

tfidf_word_sums = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
tfidf_vocab = np.array(tfidf_vectorizer.get_feature_names_out())
tfidf_top_idx = tfidf_word_sums.argsort()[::-1][:15]
top15_tfidf_words = list(zip(tfidf_vocab[tfidf_top_idx], np.round(tfidf_word_sums[tfidf_top_idx], 3)))
print(top15_tfidf_words)

## 8. BoW와 TF-IDF 문서 유사도 비교하기

1. 비교 기준 문서 인덱스를 `target_idx = 10` 으로 설정하기
2. `bow_matrix`와 `tfidf_matrix` 각각에 대해 cosine similarity를 계산하기
3. 자기 자신을 제외한 상위 5개 유사 문서를 찾기
4. 기준 문서와 함께 결과를 출력하기
5. BoW와 TF-IDF 결과가 어떻게 다른지 확인하기

In [ ]:
target_idx = 10

bow_sim_scores = cosine_similarity(bow_matrix[target_idx], bow_matrix).flatten()
tfidf_sim_scores = cosine_similarity(tfidf_matrix[target_idx], tfidf_matrix).flatten()

bow_top_doc_idx = bow_sim_scores.argsort()[::-1][1:6]
tfidf_top_doc_idx = tfidf_sim_scores.argsort()[::-1][1:6]

print('기준 문서:')
print(df_tokens.loc[target_idx, 'document'])
print()

print('[BoW 유사 문서]')
for idx in bow_top_doc_idx:
    print(f'- ({bow_sim_scores[idx]:.4f}) {df_tokens.loc[idx, "document"]}')

print()
print('[TF-IDF 유사 문서]')
for idx in tfidf_top_doc_idx:
    print(f'- ({tfidf_sim_scores[idx]:.4f}) {df_tokens.loc[idx, "document"]}')

## 9. Word2Vec으로 단어 임베딩 학습하기

1. `df_tokens['tokens']`를 학습 데이터로 사용하기
2. `vector_size=100`, `window=5`, `min_count=3`, `workers=4`, `sg=1`로 설정하기
3. 모델을 `w2v_model`로 저장하기
4. `'연기'`, `'재미'`, `'감동'`과 유사한 단어를 확인하기
5. 단어가 vocabulary에 없을 수도 있으므로 예외 없이 동작하도록 작성하기

In [ ]:
w2v_model = Word2Vec(
    sentences=df_tokens['tokens'].tolist(),
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    sg=1,
    epochs=10
)

for query in ['연기', '재미', '감동']:
    if query in w2v_model.wv:
        print(query, '->', w2v_model.wv.most_similar(query, topn=5))
    else:
        print(query, '-> vocabulary에 없음')


## 10. 평균 임베딩으로 문장 벡터 만들기

1. 토큰 리스트를 입력받아 평균 임베딩 벡터를 반환하는 `sentence_vector()` 함수 만들기
2. 단어가 하나도 없으면 0벡터를 반환하기
3. `w2v_sentence_vectors` 배열 만들기
4. 기준 문서와 가장 유사한 상위 5개 문서를 찾기
5. 결과를 출력하고 TF-IDF 유사도 결과와 비교하기

In [ ]:
def sentence_vector(tokens, model):
    valid_vecs = [model.wv[token] for token in tokens if token in model.wv]
    if not valid_vecs:
        return np.zeros(model.vector_size)
    return np.mean(valid_vecs, axis=0)

w2v_sentence_vectors = np.vstack(df_tokens['tokens'].apply(lambda x: sentence_vector(x, w2v_model)).to_list())
w2v_sim_scores = cosine_similarity([w2v_sentence_vectors[target_idx]], w2v_sentence_vectors).flatten()
w2v_top_doc_idx = w2v_sim_scores.argsort()[::-1][1:6]

print('기준 문서:')
print(df_tokens.loc[target_idx, 'document'])
print()
print('[Word2Vec 평균 임베딩 유사 문서]')
for idx in w2v_top_doc_idx:
    print(f'- ({w2v_sim_scores[idx]:.4f}) {df_tokens.loc[idx, "document"]}')


## 11. FastText로 오탈자·변형 표현 확인하기

1. `FastText` 모델을 `vector_size=100`, `window=5`, `min_count=3`, `workers=4`, `sg=1`로 학습하기
2. 모델 이름을 `ft_model`로 저장하기
3. `'재밌다'`, `'재밋다'`, `'꿀잼'`, `'최고'`, `'감동'`의 벡터를 사용할 수 있는지 확인하기
4. `FastText`에서 각 단어와 가장 유사한 단어 5개를 확인하기
5. `Word2Vec`과 비교했을 때 어떤 차이가 보이는지 확인하기

In [ ]:
ft_model = FastText(
    sentences=df_tokens['tokens'].tolist(),
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    sg=1,
    epochs=10
)

for query in ['재밌다', '재밋다', '꿀잼', '최고', '감동']:
    print(f'[{query}]')
    print(ft_model.wv.most_similar(query, topn=5))
    print()


## 12. FastText 유사도 행렬 확인하기

1. 비교할 단어 목록을 `compare_words`로 만들기
2. `FastText` 벡터를 사용해 단어 간 cosine similarity 행렬을 만들기
3. 데이터프레임으로 보기 좋게 출력하기
4. `재밌다`와 `재밋다`의 유사도가 높은지 확인하기
5. 신조어 또는 변형 표현이 어떤 방식으로 처리되는지 해석하기

In [ ]:
compare_words = ['재밌다', '재밋다', '꿀잼', '최고', '감동']
ft_vectors = np.vstack([ft_model.wv[word] for word in compare_words])
ft_sim_matrix = cosine_similarity(ft_vectors)
ft_sim_df = pd.DataFrame(ft_sim_matrix, index=compare_words, columns=compare_words)
ft_sim_df.round(4)